# Нейросети с нуля: минимум для курса по RL

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IlyaChichkanov/Reinforcement-learning/blob/main/01-intro/seminar/dl_basics.ipynb)

Этот ноутбук — для тех, кто не работал с нейросетями или подзабыл, что это такое. Он **не заменяет** курс по deep learning: здесь ровно то, без чего с недели 5 не понять, что происходит в лекциях. Читается за 2–3 часа, лучше в два захода. После него — `pytorch_intro.ipynb` (механика PyTorch, которая нужна именно в RL).

Что понадобится заранее: производная и градиент, умножение матриц, numpy.

Где это пригодится в курсе:

| Неделя | Что делаем | Что из этого ноутбука нужно |
|---|---|---|
| 5 | Deep Cross-Entropy: политика — нейросеть, обучение на элитных парах (состояние, действие) | классификация, кросс-энтропия (раздел 5) |
| 6 | DQN: сеть предсказывает ценность действия | регрессия, MSE, градиентный спуск (разделы 1–4) |
| 7 | Policy gradient: сеть выдаёт распределение над действиями | softmax, логарифм вероятности (раздел 5) |

План:

1. Модель — это функция с параметрами
2. Градиентный спуск
3. Зачем нужна нелинейность: первая нейросеть в numpy и backprop
4. То же самое на PyTorch
5. Классификация: softmax и кросс-энтропия — мостик к методу Cross-Entropy
6. Что ещё нужно знать (коротко)
7. Дорожная карта к неделе 5
8. Упражнения с проверками

In [ ]:
# Если ноутбук открыт в Google Colab: ставим недостающие пакеты. Локально (после uv sync) ячейка ничего не делает.
import importlib.util, subprocess, sys
if importlib.util.find_spec("gymnasium") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gymnasium[toy-text,classic-control]"], check=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
rng = np.random.default_rng(0)
print("torch", torch.__version__)

## 1. Модель — это функция с параметрами

Любое обучение начинается одинаково. Есть данные: входы $x_i$ и правильные ответы $y_i$. Есть **модель** — функция $f_\theta(x)$ с настраиваемыми числами $\theta$ (параметрами). Есть **функция потерь** — число, которое говорит, насколько предсказания $f_\theta(x_i)$ далеки от ответов $y_i$. Обучение — подобрать $\theta$, чтобы потери были как можно меньше.

Самая простая модель — прямая: $f_\theta(x) = w x + b$, параметров два. Потери — среднеквадратичная ошибка (MSE):

$$
L(w, b) = \frac{1}{n} \sum_{i=1}^{n} \big(w x_i + b - y_i\big)^2 .
$$

In [ ]:
x_lin = rng.uniform(-1, 1, 60)
y_lin = 2 * x_lin - 1 + rng.normal(0, 0.15, 60)      # «истина»: w = 2, b = -1, плюс шум

def predict(w, b, x):
    return w * x + b

def mse(w, b, x, y):
    return np.mean((predict(w, b, x) - y) ** 2)

for w, b in [(0.0, 0.0), (1.0, -0.5), (2.0, -1.0)]:
    print(f"w = {w:4.1f}, b = {b:5.1f}: MSE = {mse(w, b, x_lin, y_lin):.3f}")

plt.scatter(x_lin, y_lin, s=12, label="данные")
xx = np.linspace(-1, 1, 2)
plt.plot(xx, predict(1.0, -0.5, xx), "--", label="w=1, b=-0.5")
plt.plot(xx, predict(2.0, -1.0, xx), label="w=2, b=-1"); plt.legend(); plt.show()

Два параметра можно подобрать руками. У нейросети, которая играет в Atari, их миллионы — нужен автоматический способ двигать параметры в сторону меньших потерь. Это градиентный спуск.

## 2. Градиентный спуск

**Градиент** $\nabla_\theta L$ — вектор частных производных потерь по параметрам. Он показывает, в какую сторону потери растут быстрее всего, значит, шаг **против** градиента их уменьшает:

$$
\theta \leftarrow \theta - \eta \, \nabla_\theta L, \qquad \eta - \text{learning rate (шаг)}.
$$

Для прямой градиент считается руками:

$$
\frac{\partial L}{\partial w} = \frac{2}{n} \sum_i (w x_i + b - y_i)\, x_i, \qquad
\frac{\partial L}{\partial b} = \frac{2}{n} \sum_i (w x_i + b - y_i).
$$

In [ ]:
def grad_mse(w, b, x, y):
    err = predict(w, b, x) - y
    return 2 * np.mean(err * x), 2 * np.mean(err)

def gradient_descent(lr=0.3, steps=60, w=0.0, b=0.0):
    path, losses = [(w, b)], [mse(w, b, x_lin, y_lin)]
    for _ in range(steps):
        dw, db = grad_mse(w, b, x_lin, y_lin)
        w, b = w - lr * dw, b - lr * db
        path.append((w, b)); losses.append(mse(w, b, x_lin, y_lin))
    return np.array(path), np.array(losses)

path, losses = gradient_descent()
print(f"после {len(losses) - 1} шагов: w = {path[-1, 0]:.3f}, b = {path[-1, 1]:.3f}, MSE = {losses[-1]:.4f}")

# Поверхность потерь и путь спуска по ней.
W, B = np.meshgrid(np.linspace(-1, 4, 80), np.linspace(-3, 1.5, 80))
L = np.array([[mse(w_, b_, x_lin, y_lin) for w_ in W[0]] for b_ in B[:, 0]])
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].contour(W, B, L, levels=30); axes[0].plot(path[:, 0], path[:, 1], "o-", ms=3, color="C3")
axes[0].set_xlabel("w"); axes[0].set_ylabel("b"); axes[0].set_title("поверхность потерь и путь спуска")
for lr in [0.02, 0.3, 1.05]:
    axes[1].plot(gradient_descent(lr=lr)[1][:40], label=f"lr = {lr}")
axes[1].set_yscale("log"); axes[1].set_xlabel("шаг"); axes[1].set_ylabel("MSE"); axes[1].legend(); axes[1].set_title("learning rate")
plt.show()

Три вещи, которые видно на картинке и которые будут преследовать вас весь курс:

* **learning rate** — главный гиперпараметр: слишком маленький — учимся вечно, слишком большой — расходимся;
* потери убывают **не до нуля**: в данных шум, идеальной прямой нет;
* всё, что нужно для обучения, — уметь считать градиент. Для прямой мы вывели его руками; для сети с миллионом параметров это сделает за нас autograd (раздел 4).

## 3. Зачем нужна нелинейность: первая нейросеть

Прямая — плохая модель почти для всего. Возьмём задачу, где ответ — класс (0 или 1), а точки лежат двумя спиралями: никакая прямая их не разделит.

In [ ]:
def make_spirals(n=300, noise=0.15, seed=0):
    rng = np.random.default_rng(seed)
    t = np.linspace(0.4, 3.0, n)
    X, y = [], []
    for k in range(2):
        angle = t * np.pi + k * np.pi
        X.append(np.stack([t * np.cos(angle), t * np.sin(angle)], 1) + rng.normal(0, noise, (n, 2))); y.append(np.full(n, k))
    X, y = np.concatenate(X).astype(np.float32), np.concatenate(y)
    idx = rng.permutation(len(y))
    return X[idx], y[idx]

X, y = make_spirals()
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=10); plt.title("две спирали"); plt.show()

**Нейросеть** — это несколько прямых (точнее, линейных отображений), между которыми стоит нелинейная функция. Двухслойная сеть с $h$ нейронами в скрытом слое:

$$
\mathbf{z} = \tanh(\mathbf{x} W_1 + \mathbf{b}_1), \qquad \hat{\mathbf{y}} = \mathbf{z} W_2 + \mathbf{b}_2 .
$$

$W_1$ — матрица $2 \times h$, $W_2$ — матрица $h \times 1$. Слово «слой» означает одно линейное отображение плюс нелинейность (**функция активации**: $\tanh$, ReLU $= \max(0, x)$, сигмоида). Без нелинейности два слоя схлопнулись бы в один: произведение матриц — снова матрица. С нелинейностью сеть может изобразить почти любую функцию, если нейронов достаточно.

Напишем такую сеть в numpy, включая **backprop** — вычисление градиента по всем параметрам. Backprop — это цепное правило дифференцирования, применённое слой за слоем в обратном порядке: сначала градиент по выходу, из него — по $W_2$ и по $\mathbf{z}$, из градиента по $\mathbf{z}$ — по $W_1$. Ничего, кроме правила «производная композиции — произведение производных», здесь нет.

Чтобы было проще, обучим её сначала на регрессии: приблизить $\sin 3x$ на $[-1, 1]$.

In [ ]:
xs = np.linspace(-1, 1, 200).reshape(-1, 1)
ys = np.sin(3 * xs)

def init(n_in, n_hidden, n_out, seed=0):
    r = np.random.default_rng(seed)
    return {"W1": r.normal(0, 0.5, (n_in, n_hidden)), "b1": np.zeros(n_hidden),
            "W2": r.normal(0, 0.5, (n_hidden, n_out)), "b2": np.zeros(n_out)}

def forward(p, x):
    z = np.tanh(x @ p["W1"] + p["b1"])                    # скрытый слой
    return z @ p["W2"] + p["b2"], z                        # выход и скрытые активации (нужны для backprop)

def backward(p, x, y, y_hat, z):
    n = len(x)
    d_out = 2 * (y_hat - y) / n                            # dL/dŷ для MSE
    grads = {"W2": z.T @ d_out, "b2": d_out.sum(0)}        # слой 2
    d_z = d_out @ p["W2"].T * (1 - z ** 2)                 # через tanh: tanh' = 1 - tanh²
    grads.update({"W1": x.T @ d_z, "b1": d_z.sum(0)})      # слой 1
    return grads

params = init(1, 32, 1)
for step in range(3000):
    y_hat, z = forward(params, xs)
    grads = backward(params, xs, ys, y_hat, z)
    for k in params:
        params[k] -= 0.05 * grads[k]                       # градиентный спуск
    if step % 1000 == 0:
        print(f"шаг {step:4d}: MSE = {np.mean((y_hat - ys) ** 2):.4f}")

plt.plot(xs, ys, label="sin 3x"); plt.plot(xs, forward(params, xs)[0], "--", label="сеть 1-32-1"); plt.legend(); plt.show()

Тридцать строк — и сеть приближает синусоиду. Всё обучение — те же три действия, что у прямой: посчитать предсказание, посчитать градиент, сделать шаг против него. Разница только в том, что градиент считается через цепочку слоёв.

Писать backprop руками для каждой архитектуры никто не хочет. Дальше — PyTorch, который делает это автоматически.

## 4. То же самое на PyTorch

PyTorch — это numpy с двумя дополнениями: **autograd** (градиенты считаются автоматически для любой цепочки операций) и GPU. Основной объект — тензор; если у тензора `requires_grad=True`, PyTorch запоминает все операции с ним, и вызов `.backward()` на итоговом числе заполняет `.grad` у всех участников.

Повторим прямую из раздела 1, но градиент за нас посчитает autograd.

In [ ]:
xt, yt = torch.tensor(x_lin, dtype=torch.float32), torch.tensor(y_lin, dtype=torch.float32)
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

for step in range(60):
    loss = ((w * xt + b - yt) ** 2).mean()               # прямой проход: строится граф вычислений
    loss.backward()                                       # обратный проход: w.grad и b.grad заполнены
    with torch.no_grad():                                 # шаг спуска не должен попадать в граф
        w -= 0.3 * w.grad; b -= 0.3 * b.grad
    w.grad.zero_(); b.grad.zero_()                        # градиенты накапливаются, поэтому обнуляем

print(f"w = {w.item():.3f}, b = {b.item():.3f}  (numpy дал {path[-1, 0]:.3f}, {path[-1, 1]:.3f})")

Для сетей есть готовые кирпичи: `nn.Linear` — слой $xW + b$ с параметрами внутри, `nn.Sequential` — цепочка слоёв, `torch.optim` — оптимизаторы (тот же градиентный спуск, но с улучшениями; `Adam` — стандартный выбор по умолчанию). Цикл обучения всегда состоит из одних и тех же четырёх строк.

In [ ]:
xs_t, ys_t = torch.tensor(xs, dtype=torch.float32), torch.tensor(ys, dtype=torch.float32)

model = nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 1))     # та же архитектура, что в numpy
opt = torch.optim.Adam(model.parameters(), lr=0.01)

for step in range(1500):
    opt.zero_grad()                       # 1. обнулить градиенты
    loss = F.mse_loss(model(xs_t), ys_t)  # 2. прямой проход и потери
    loss.backward()                       # 3. градиенты
    opt.step()                            # 4. шаг оптимизатора
    if step % 500 == 0:
        print(f"шаг {step:4d}: MSE = {loss.item():.4f}")

print("параметров в модели:", sum(p.numel() for p in model.parameters()))
plt.plot(xs, ys, label="sin 3x"); plt.plot(xs, model(xs_t).detach(), "--", label="nn.Sequential"); plt.legend(); plt.show()

Что здесь важно запомнить:

* `opt.zero_grad()` обязателен: градиенты **накапливаются** между вызовами `backward()`;
* `.detach()` (или `with torch.no_grad()`) отрезает тензор от графа — так делают всё, что не должно обучаться: рисование, логирование, целевые значения;
* `loss.item()` превращает тензор из одного числа в обычное число Python.

## 5. Классификация: softmax и кросс-энтропия

Когда ответ — класс, сеть выдаёт по одному числу на класс (**логиты**), а **softmax** превращает их в вероятности:

$$
p_k = \frac{e^{z_k}}{\sum_j e^{z_j}} .
$$

Функция потерь для классификации — **кросс-энтропия**: минус логарифм вероятности, которую сеть выдала правильному классу, усреднённый по примерам:

$$
L = -\frac{1}{n} \sum_i \log p_{y_i}(x_i).
$$

Если сеть уверена в правильном ответе ($p \to 1$), потери близки к нулю; если уверена в неправильном ($p \to 0$), потери огромны. В PyTorch это `nn.CrossEntropyLoss`: на вход логиты и номера классов, softmax внутри.

Разделим спирали.

In [ ]:
X_t, y_t = torch.tensor(X), torch.tensor(y)
X_train, y_train, X_test, y_test = X_t[:400], y_t[:400], X_t[400:], y_t[400:]      # отложенная выборка — для честной оценки

clf = nn.Sequential(nn.Linear(2, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh(), nn.Linear(64, 2))
opt = torch.optim.Adam(clf.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

for step in range(2000):
    opt.zero_grad(); loss = loss_fn(clf(X_train), y_train); loss.backward(); opt.step()

with torch.no_grad():
    acc_train = (clf(X_train).argmax(1) == y_train).float().mean().item()
    acc_test = (clf(X_test).argmax(1) == y_test).float().mean().item()
    gx, gy = np.meshgrid(np.linspace(-3.5, 3.5, 200), np.linspace(-3.5, 3.5, 200))
    grid = torch.tensor(np.stack([gx.ravel(), gy.ravel()], 1).astype(np.float32))
    pred = clf(grid).argmax(1).numpy().reshape(gx.shape)
print(f"точность: обучение {acc_train:.1%}, отложенная выборка {acc_test:.1%}")
plt.contourf(gx, gy, pred, alpha=0.25, cmap="coolwarm"); plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=8); plt.show()

### Мостик к RL

Вспомните метод Cross-Entropy из лекции 1: на каждой итерации мы брали элитные эпизоды и делали так, чтобы их действия выбирались чаще. Для таблицы «чаще» означало «посчитать частоты». Для нейросети это **ровно классификация из этого раздела**: объекты — состояния из элитных эпизодов, метки классов — действия, которые в них сделали, потери — кросс-энтропия. Сеть $\pi_\theta(a \mid s)$ выдаёт логиты по действиям, softmax превращает их в политику. Это deep Cross-Entropy Method, неделя 5, и код там будет отличаться от ячейки выше десятком строк.

## 6. Что ещё нужно знать (коротко)

* **Батчи.** Считать потери по всем данным сразу дорого; обычно берут случайную порцию (mini-batch, 32–256 примеров) и делают шаг по ней. Градиент получается шумным, но шагов можно сделать гораздо больше. В RL батч — это порция переходов из памяти агента.
* **Переобучение и валидация.** Сеть с большим числом параметров может выучить обучающие данные наизусть. Поэтому качество всегда меряют на отложенных данных (как `X_test` выше), а не на тех, на которых учили. В RL «отложенные данные» — это новые эпизоды.
* **`model.train()` / `model.eval()` и `torch.no_grad()`.** Некоторые слои (dropout, batch norm) ведут себя по-разному при обучении и применении; всё, что не обучается (оценка, отрисовка, целевые значения), делают без графа вычислений.
* **GPU.** `model.to("cuda")` и `x.to("cuda")` — и всё считается на видеокарте. Для этого курса до недели 5 включительно хватает CPU.
* **Сиды.** `torch.manual_seed(0)` — чтобы эксперимент повторялся. В RL результаты **очень** сильно зависят от сида, поэтому кривые всегда строят по нескольким запускам.
* **Типы.** Нейросети считают в `float32`; numpy по умолчанию даёт `float64`, и при смешивании PyTorch ругается. `torch.as_tensor(x, dtype=torch.float32)` решает почти все такие проблемы.

## 7. Дорожная карта к неделе 5

Если нейросети для вас новы, вот план на четыре недели, по одному пункту в неделю (каждый — 2–3 часа):

1. **Неделя 1.** Этот ноутбук, разделы 1–3, плюс первые две серии [3Blue1Brown «Neural networks»](https://www.3blue1brown.com/topics/neural-networks) — лучшая визуальная интуиция про слои и градиентный спуск.
2. **Неделя 2.** Разделы 4–5 здесь и [PyTorch «Learn the Basics»](https://pytorch.org/tutorials/beginner/basics/intro.html) — тензоры, autograd, `nn.Module`, цикл обучения.
3. **Неделя 3.** `pytorch_intro.ipynb` из этой же папки и лекция 1 из [Karpathy, «Neural Networks: Zero to Hero»](https://karpathy.ai/zero-to-hero.html) — backprop с нуля на 100 строках, после неё autograd перестаёт быть магией.
4. **Неделя 4.** Упражнения из раздела 8 (они же бонусная часть ДЗ 1) и глава про нейронные сети из [учебника ШАД по машинному обучению](https://education.yandex.ru/handbook/ml) — на русском и с нужной строгостью.

Если хочется больше: [Deep Learning School](https://dls.samcs.ru/) (МФТИ, бесплатный курс с видео и заданиями), Goodfellow, Bengio, Courville, [*Deep Learning*](https://www.deeplearningbook.org/), глава 6.

## 8. Упражнения с проверками

Каждое упражнение проверяется `assert`. Первые три — это бонусная часть ДЗ 1.

**8.1.** Градиент руками: для прямой $\hat y = w x + b$ и MSE верните $(\partial L/\partial w, \partial L/\partial b)$ и сравните с autograd.

In [ ]:
def grad_manual(w, b, x, y):
    # TODO: ваш код здесь
    raise NotImplementedError

w0, b0 = 0.5, 0.0
w_ = torch.tensor(w0, requires_grad=True); b_ = torch.tensor(b0, requires_grad=True)
((w_ * xt + b_ - yt) ** 2).mean().backward()
dw, db = grad_manual(w0, b0, x_lin, y_lin)
assert np.isclose(dw, w_.grad.item(), atol=1e-5) and np.isclose(db, b_.grad.item(), atol=1e-5)
print("ok")

**8.2.** Регрессия: обучите сеть из двух скрытых слоёв по 32 нейрона приближать $\sin 3x$ на $[-1, 1]$ так, чтобы MSE на сетке из 200 точек была меньше 0.01.

In [ ]:
# TODO: модель и обучение
model2 = ...

with torch.no_grad():
    mse2 = F.mse_loss(model2(xs_t), ys_t).item()
print(f"MSE = {mse2:.4f}")
assert mse2 < 0.01

**8.3.** Классификация: обучите свой классификатор спиралей с точностью не ниже 95% на отложенной выборке. Попробуйте меньше нейронов (например, 8) — с какого размера сеть перестаёт справляться?

In [ ]:
# TODO: модель и обучение
clf2 = ...

with torch.no_grad():
    acc2 = (clf2(X_test).argmax(1) == y_test).float().mean().item()
print(f"точность на отложенной выборке: {acc2:.1%}")
assert acc2 >= 0.95

**8.4.** Softmax и кросс-энтропия руками. Реализуйте `cross_entropy(logits, labels)` через softmax и логарифм так, чтобы результат совпадал с `F.cross_entropy`. Подсказка: перед экспонентой вычтите из логитов максимум по строке — иначе на больших логитах будет переполнение (сам softmax от этого не меняется).

In [ ]:
def cross_entropy(logits, labels):
    # logits: (n, k) numpy, labels: (n,) номера классов; вернуть среднюю кросс-энтропию
    # TODO: ваш код здесь
    raise NotImplementedError

logits = rng.normal(0, 3, (5, 4)); labels = np.array([0, 3, 1, 2, 3])
ref = F.cross_entropy(torch.tensor(logits), torch.tensor(labels)).item()
assert np.isclose(cross_entropy(logits, labels), ref, atol=1e-6)
assert np.isfinite(cross_entropy(logits * 1000, labels)), "на больших логитах не должно быть переполнения"
print("ok")

**8.5.** Прямой проход руками. Возьмите обученный `clf` из раздела 5, достаньте его веса (`clf[0].weight`, `clf[0].bias`, …) и посчитайте выход сети для `X_test` в numpy: два линейных слоя с $\tanh$ между ними и линейный выход. Помните, что `nn.Linear` хранит матрицу в форме `(out, in)`, то есть считает $x W^\top + b$.

In [ ]:
def forward_numpy(clf, X):
    # TODO: ваш код здесь
    raise NotImplementedError

out = forward_numpy(clf, X_test.numpy())
assert np.allclose(out, clf(X_test).detach().numpy(), atol=1e-4)
print("ok")